# Building a Fully Local AI Coding Assistant on Windows

## llama.cpp + Qwen3-4B GGUF + llama-server + Continue + VS Code

**Status:** Reconstructed from the complete setup process\
**Platform:** Windows\
**Hardware:** \~16 GB RAM, Intel Iris Xe Graphics\
**Model:** Qwen3-4B-Q4_K_M.gguf\
**Inference runtime:** llama.cpp\
**IDE:** Visual Studio Code\
**VS Code AI extension:** Continue\
**Cloud inference:** None\
**Ollama:** Not used\
**OpenRouter:** Not used for inference

------------------------------------------------------------------------


# 1. Objective

The goal was to build a local AI coding assistant with as few
abstraction layers as practical.

The desired architecture was:

``` text
VS Code
   |
   v
Continue
   |
   | HTTP localhost
   v
llama-server
   |
   v
Qwen3-4B-Q4_K_M.gguf
   |
   v
Local CPU / RAM / Intel Iris Xe
```

The important requirement was **not to use Ollama or an external
inference provider as a middleman**.

There are two useful modes in the final setup:

### Direct terminal inference

``` text
CMD
 |
 v
llama-cli.exe
 |
 v
Qwen3 GGUF
 |
 v
Local hardware
```

### VS Code inference

``` text
VS Code
 |
 v
Continue
 |
 v
http://127.0.0.1:8080
 |
 v
llama-server.exe
 |
 v
Qwen3 GGUF
 |
 v
Local hardware
```

`llama-server.exe` is not Ollama. It is part of the llama.cpp project
itself and provides the local HTTP interface required by applications
such as Continue.

------------------------------------------------------------------------


# 2. Hardware assessment — make the setup hardware-agnostic

The original guide was written around one Windows machine:

```text
~16 GB RAM
Intel Iris Xe Graphics
x64 Windows
MSVC
```

That is useful as a worked example, but it should **not** determine the commands for every other machine.

llama.cpp supports multiple CPU architectures and acceleration backends. The current official build documentation lists CPU, Metal, SYCL, CUDA, HIP, Vulkan, OpenCL and other backends, while the feature matrix covers CPU AVX/AVX2, ARM NEON, Metal, CUDA, ROCm, SYCL and Vulkan. citeturn0search0turn0search8

Therefore, use the hardware branch below **before** choosing your CMake command.

## 2.1 Hardware decision table

| Hardware | Recommended starting backend | Typical model range* | Build approach |
|---|---|---:|---|
| Intel/AMD x86-64 CPU only | CPU | 1B–32B depending on RAM | Normal CMake |
| Older x86 CPU | CPU | 1B–7B | Normal CMake; verify AVX support |
| Modern Intel/AMD CPU | CPU | 3B–32B depending on RAM | Native CPU build |
| Intel Arc / supported Intel GPU | Vulkan or SYCL | 3B–32B+ depending on VRAM/RAM | Vulkan or Intel oneAPI/SYCL |
| Intel integrated GPU | CPU first; Vulkan/SYCL experiment | 1B–8B typically | CPU baseline + optional GPU backend |
| NVIDIA GPU | CUDA | VRAM-dependent | CUDA build |
| AMD GPU | HIP/ROCm or Vulkan | VRAM-dependent | HIP/ROCm or Vulkan |
| Apple Silicon | Metal | Unified-memory-dependent | Metal build |
| ARM64 Linux/Windows | ARM CPU/NEON | RAM-dependent | ARM64 build |
| Raspberry Pi / ARM SBC | CPU/NEON, optional Vulkan depending on device | 1B–4B typically | ARM64 CPU build |
| Large-memory workstation/server | CPU + optional accelerator | 14B–70B+ | CPU or accelerator build |

\*Model-size ranges are practical starting points, not hard limits. Actual usable size depends on RAM/VRAM, quantization, context length, architecture and memory bandwidth.

## 2.2 RAM-based model sizing

Use this only as a starting point:

```text
8 GB RAM
 └── 1B–3B Q4

16 GB RAM
 └── 3B–8B Q4

32 GB RAM
 └── 7B–14B Q4/Q5

64 GB RAM
 └── 14B–32B Q4/Q5

128 GB RAM
 └── 32B–70B Q4/Q5

256 GB+
 └── 70B+ / very large models
```

Model file size is **not** the same as total RAM requirement. Runtime buffers, KV cache, context length, operating-system memory and other allocations also matter.

## 2.3 CPU instruction sets

For x86 CPUs, llama.cpp can use optimized instruction paths such as:

```text
AVX
AVX2
FMA
F16C
AVX512
```

For ARM CPUs, NEON/ARM-specific optimizations are relevant.

Do not select a build only because a CPU has many cores. For LLM inference, memory bandwidth, vector instructions, cache behavior and model characteristics can be just as important.

## 2.4 Native vs portable CPU build

There are two important CPU-build choices.

### Option A — Native build

Build for the CPU currently running the build:


In [ ]:
!cmake -B build
!cmake --build build --config Release -j


This is the normal choice for a machine where the binary will stay on that machine.

### Option B — More portable CPU build

If the same binary needs to run across different x86-64 CPUs, disable native CPU-specific compilation:


In [ ]:
!cmake -B build -DGGML_NATIVE=OFF
!cmake --build build --config Release -j


For official multi-variant Windows builds, llama.cpp's own CI also uses `GGML_NATIVE=OFF` and can build all CPU variants. citeturn0search2

### Important

Do **not** blindly add `-DGGML_NATIVE=OFF` to a performance-focused single-machine build. Native compilation can enable CPU-specific optimizations.

## 2.5 Backend selection rule

Use:

```text
CPU only
  → normal CPU build

NVIDIA
  → CUDA

AMD
  → HIP/ROCm where supported
  → Vulkan as an alternative

Intel GPU
  → Vulkan
  → SYCL/oneAPI where appropriate

Apple Silicon
  → Metal

ARM CPU
  → ARM64/NEON CPU build
```

The current llama.cpp feature matrix shows that K-quants are supported across CPU, ARM NEON, Metal, CUDA, ROCm, SYCL and Vulkan, with backend-specific differences in performance/features. citeturn0search8


# 3. Why GGUF was used

`llama.cpp` works particularly well with GGUF model files.

A GGUF file contains the model data in a format that llama.cpp can load
directly.

The important distinction is:

``` text
Model source
     |
     v
GGUF weights
     |
     v
llama.cpp
```

No model manager is required.

The model file itself is separate from the llama.cpp software.

------------------------------------------------------------------------


# 4. Initial mistake: using git pull instead of git clone

The first command attempted was:

``` cmd
git pull https://github.com/ggml-org/llama.cpp.git
```

This produced:

``` text
fatal: not a git repository (or any of the parent directories): .git
```

## Why it happened

`git pull` is used to update an existing Git repository.

At that point, `C:\Users\SANTHOSH JAPALA` was not a cloned llama.cpp
repository.

## Correct solution

For a first download, use `git clone`:

``` cmd
cd C:\
git clone https://github.com/ggml-org/llama.cpp.git
```

Then:

``` cmd
cd C:\llama.cpp
```

After cloning, `git pull` becomes appropriate:

``` cmd
cd C:\llama.cpp
git pull
```

### Rule to remember

``` text
First download:
git clone

Later update:
git pull
```

------------------------------------------------------------------------


# 5. CMake was missing

After entering:

``` cmd
C:\llama.cpp
```

the build was attempted.

Initially:

``` cmd
cmake -B build
```

could not be used because CMake was not installed/available.

## Solution

Install CMake from the official CMake website.

After installation, close the existing terminal and open a new one.

Verify:

``` cmd
cmake --version
```

A version number confirms that CMake is available through PATH.

## Important lesson

When installing command-line development tools on Windows:

1.  install the tool;
2.  close the old terminal;
3.  open a new terminal;
4.  run the version command again.

The old terminal may not have the updated PATH.

------------------------------------------------------------------------


# 6. CMake existed, but the C/C++ compiler was missing

After CMake became available, the following was attempted:

``` cmd
cmake -B build
```

CMake reported:

``` text
CMake Error at CMakeLists.txt:2 (project):
  Running

   'nmake' '-?'

  failed with:

   no such file or directory

CMake Error: CMAKE_C_COMPILER not set, after EnableLanguage
CMake Error: CMAKE_CXX_COMPILER not set, after EnableLanguage
```

## Meaning

CMake itself was working.

The problem was the compiler toolchain.

CMake needs:

-   C compiler;
-   C++ compiler;
-   linker;
-   Windows SDK;
-   build environment.

CMake is a build-system generator. It is not itself the C++ compiler.

------------------------------------------------------------------------


# 7. VS Code is not the C++ compiler

At one point it was noted that Visual Studio Code was already installed.

That is fine, but VS Code alone does not provide the MSVC compiler.

The distinction is:

``` text
VS Code
= editor / development environment

MSVC
= C/C++ compiler and linker

CMake
= build-system configuration/generation tool
```

Therefore, keeping VS Code was completely fine, but a Microsoft C++
toolchain still had to be installed.

------------------------------------------------------------------------


# 8. Installing Visual Studio Build Tools

The full Visual Studio IDE was not necessary.

The required component was **Visual Studio Build Tools**.

In Visual Studio Installer, the important workload was:

``` text
Desktop development with C++
```

The installation included:

``` text
MSVC C++ Build Tools
Windows SDK
C++ CMake tools for Windows
```

This provided the actual compiler needed to build llama.cpp.

## Why the IDE was confusing

The Microsoft download page prominently showed:

``` text
Visual Studio Community
Visual Studio Professional
Visual Studio Enterprise
```

Those are IDE distributions.

The requirement here was the Build Tools workload, not the full IDE.

The Visual Studio Installer/Build Tools installation can still be used
without doing day-to-day development inside Visual Studio.

------------------------------------------------------------------------


# 9. Problem: Developer Command Prompt was not visible in Start

After installing Build Tools, the expected Developer Command Prompt
shortcut was not immediately visible.

Several attempts to locate it were made.

One attempt used:

``` cmd
vswhere
```

but Windows returned:

``` text
'vswhere' is not recognized
```

## Resolution

`vswhere` was not necessary.

The Visual Studio installation directories were located manually.

PowerShell was used to inspect:

``` powershell
Get-ChildItem "C:\Program Files (x86)\Microsoft Visual Studio" -Directory
```

This revealed:

``` text
18
Installer
Shared
```

The Visual Studio 2026/18 Build Tools installation was therefore under
the x86 Program Files tree.

------------------------------------------------------------------------


# 10. Problem: CMD syntax was used inside PowerShell

A command was attempted like:

``` cmd
dir "C:\Program Files\Microsoft Visual Studio" /s /b
```

but the terminal was PowerShell.

PowerShell interpreted `dir` as its `Get-ChildItem` command, resulting
in:

``` text
Get-ChildItem : A positional parameter cannot be found that accepts argument '/b'.
```

## Why

CMD and PowerShell have different command syntax.

For example:

``` text
CMD:
dir /s /b

PowerShell:
Get-ChildItem -Recurse
```

## Correct PowerShell equivalent

``` powershell
Get-ChildItem "C:\Program Files\Microsoft Visual Studio" -Filter VsDevCmd.bat -Recurse -ErrorAction SilentlyContinue | Select-Object -ExpandProperty FullName
```

This distinction is important when troubleshooting Windows development
environments.

------------------------------------------------------------------------


# 11. Loading the Visual Studio compiler environment

The important requirement was to make this command work:

``` cmd
cl
```

Once correctly initialized, it produced:

``` text
Microsoft (R) C/C++ Optimizing Compiler
Version 19.51.36252
```

That confirmed MSVC was available.

The x64 development environment was loaded using:

``` cmd
VsDevCmd.bat -arch=x64
```

The compiler path ultimately showed an x64 toolchain such as:

``` text
Hostx64\x64\cl.exe
```

This is important because the intended llama.cpp build was x64.

------------------------------------------------------------------------


# 12. Re-running CMake after the compiler was available

At this point, choose **one build branch** based on the hardware.

Do not run every branch.

## 12.1 Windows x64 — CPU-only, recommended baseline

From:


In [ ]:
!C:\llama.cpp>


clean any previous failed configuration:


In [ ]:
!rmdir /s /q build


Then:


In [ ]:
!cmake -B build


Build:


In [ ]:
!cmake --build build --config Release -j


This is the current official llama.cpp CPU build pattern. citeturn0search0

If `-j` is rejected by an older CMake/build environment, use:


In [ ]:
!cmake --build build --config Release


## 12.2 Windows x64 — portable CPU build

If the binary must work across multiple x86-64 CPUs:


In [ ]:
!rmdir /s /q build
!cmake -B build -DGGML_NATIVE=OFF
!cmake --build build --config Release -j


For maximum CPU-variant coverage, llama.cpp's own Windows CI uses `GGML_NATIVE=OFF` with `GGML_CPU_ALL_VARIANTS=ON` in its multi-variant build. citeturn0search2

A more portable/all-variants build is:


In [ ]:
!rmdir /s /q build
!cmake -B build -DGGML_NATIVE=OFF -DGGML_CPU_ALL_VARIANTS=ON
!cmake --build build --config Release -j


Use this when portability matters more than keeping the build as simple as possible.

## 12.3 Windows x64 — NVIDIA CUDA

Install the appropriate NVIDIA CUDA Toolkit first.

Then, from an NVIDIA-capable developer environment:


In [ ]:
!rmdir /s /q build
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j


The official llama.cpp build guide uses `-DGGML_CUDA=ON`. citeturn0search0

If you need a binary intended to be less tied to the GPU present during compilation, consult the current CUDA/non-native section of the official build guide before choosing additional flags.

## 12.4 Windows x64 — Vulkan

Install the Vulkan SDK and verify:


In [ ]:
!vulkaninfo


Then:


In [ ]:
!rmdir /s /q build
!cmake -B build -DGGML_VULKAN=ON
!cmake --build build --config Release -j


The official llama.cpp Windows Vulkan instructions use `-DGGML_VULKAN=ON`. citeturn0search0

After building, a Vulkan-enabled CLI can be tested with GPU-layer offloading:


In [ ]:
!build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -ngl 99 -cnv


`-ngl 99` requests aggressive GPU-layer offloading; the actual amount depends on available GPU memory and the model.

## 12.5 Windows x64 — Intel GPU with SYCL/oneAPI

This is a separate Intel-specific path.

Install Intel oneAPI and initialize it:


In [ ]:
!call "C:\Program Files (x86)\Intel\oneAPI\setvars.bat" intel64 --force


Then:


In [ ]:
!cd /d C:\llama.cpp
!rmdir /s /q build
!cmake -B build -G "Ninja" -DGGML_SYCL=ON -DCMAKE_C_COMPILER=cl -DCMAKE_CXX_COMPILER=icx -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j


The official llama.cpp Windows SYCL workflow uses Intel oneAPI, `cl`, `icx`, Ninja and `GGML_SYCL=ON`. citeturn0search4turn0search6

Do not use this branch merely because a machine has an Intel CPU. It is for Intel GPU/SYCL acceleration.

## 12.6 Windows ARM64

For Windows on ARM64, use the official ARM64 preset rather than copying x64 compiler paths:


In [ ]:
!cmake --preset arm64-windows-llvm-release -DGGML_OPENMP=OFF
!cmake --build build-arm64-windows-llvm-release


The current official build documentation provides an ARM64 Windows path. citeturn0search0

## 12.7 Apple Silicon

Do not use the Windows commands in this guide.

For macOS with Apple Silicon, use the Metal build documented by llama.cpp:


In [ ]:
!cmake -B build -DGGML_METAL=ON
!cmake --build build --config Release


Then use the resulting `llama-cli`/`llama-server` binaries from the macOS build directory.

Always check the current official macOS/Metal section because the exact build environment may change. citeturn0search0

## 12.8 Linux x86-64 CPU

The basic CPU build is:


In [ ]:
!cmake -B build
!cmake --build build --config Release -j


For a Linux distribution with a package-managed llama.cpp, using the distribution package may be easier, but building from source gives you direct control over backends and build options.

## 12.9 Linux ARM64

Use the ARM/NEON CPU path appropriate for the distribution and CPU.

The core build remains:


In [ ]:
!cmake -B build
!cmake --build build --config Release -j


Then verify that llama.cpp reports the expected ARM CPU capabilities.

## 12.10 What not to do

Do not combine unrelated backend flags such as:

```text
GGML_CUDA
GGML_VULKAN
GGML_SYCL
GGML_HIP
GGML_METAL
```

without a specific reason.

Choose the backend that corresponds to the hardware and operating system.


# 13. Building llama.cpp

The build command is now hardware-dependent.

For the **original Windows x64 CPU baseline**, use:


In [ ]:
!cd /d C:\llama.cpp
!cmake --build build --config Release -j


For **NVIDIA CUDA**, use the CUDA-configured build directory.

For **Vulkan**, use the Vulkan-configured build directory.

For **Intel SYCL**, use the Ninja/SYCL build directory.

For **Windows ARM64**, use the ARM64 preset directory.

The important rule is:

```text
Configure once for the chosen backend
              ↓
Build that configuration
              ↓
Test the resulting binary
```

Do not repeatedly reconfigure the same `build` directory from CPU → CUDA → Vulkan → SYCL.

Instead, delete the build directory or use separate build directories.

### Recommended separate build directories

For experiments:

```text
build-cpu
build-cpu-portable
build-vulkan
build-cuda
build-sycl
```

For example:


In [ ]:
!cmake -B build-vulkan -DGGML_VULKAN=ON
!cmake --build build-vulkan --config Release -j


Then:


In [ ]:
!build-vulkan\bin\Release\llama-cli.exe --version


This preserves the CPU build.


# 14. Why npm appeared during the build

The build output included:

``` text
-- UI: running npm ci
```

This is related to llama.cpp's included UI/tooling assets.

It does not mean an external AI model was being installed.

The build was provisioning the project's web/UI assets.

Because the main requirement was command-line inference, the UI itself
was not part of the intended inference architecture.

The safest approach at that point was to let the current build complete
instead of interrupting it.

------------------------------------------------------------------------


# 15. Verifying llama-cli

After compilation:

``` cmd
dir build\bin\Release\llama-cli.exe
```

The executable was found at:

``` text
C:\llama.cpp\build\bin\Release\llama-cli.exe
```

Then:

``` cmd
build\bin\Release\llama-cli.exe --version
```

returned:

``` text
version: 10358 (030ebb558)
built with MSVC 19.51.36252.0 for x64
```

This confirmed that llama.cpp had successfully compiled.

------------------------------------------------------------------------


# 16. Checking the GPU

The GPU was checked with PowerShell:

``` powershell
Get-CimInstance Win32_VideoController | Select-Object Name
```

Result:

``` text
Intel(R) Iris(R) Xe Graphics
```

This meant CUDA was not appropriate because CUDA is an NVIDIA
technology.

The initial inference setup was therefore kept CPU-first.

Vulkan acceleration can be investigated later for Intel GPU usage.

------------------------------------------------------------------------


# 17. Checking system RAM

The command used was:

``` cmd
wmic computersystem get TotalPhysicalMemory
```

It returned:

``` text
16849293312
```

This is approximately 16 GB.

That confirmed that Qwen3-4B Q4_K_M was a reasonable starting model.

------------------------------------------------------------------------


# 18. Creating the model directory

A dedicated directory was created:

``` cmd
mkdir C:\llama.cpp\models
```

The resulting structure was:

``` text
C:\llama.cpp
│
├── build
│   └── bin
│       └── Release
│           └── llama-cli.exe
│
├── models
│   └── Qwen3-4B-Q4_K_M.gguf
│
└── ...
```

Keeping model files separate from build artifacts makes the installation
easier to manage.

------------------------------------------------------------------------


# 19. Downloading the GGUF model directly

The model was downloaded using `curl`.

From:

``` cmd
C:\llama.cpp\models>
```

the command was:

``` cmd
curl -L -o Qwen3-4B-Q4_K_M.gguf "https://huggingface.co/Qwen/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf?download=true"
```

The download reached approximately:

``` text
2.32G
```

and completed successfully.

## Important

The `-L` option follows redirects.

The small initial:

``` text
100 1008
```

was not the model.

The subsequent:

``` text
100 2.32G
```

was the actual model download.

------------------------------------------------------------------------


# 20. Problem: model command said "path not found"

The command was:

``` cmd
build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -cnv
```

but the prompt showed:

``` text
C:\llama.cpp\models>
```

Therefore the relative path:

``` text
build\bin\Release\...
```

was being interpreted relative to:

``` text
C:\llama.cpp\models
```

which was wrong.

## Resolution

Move back to the repository root:

``` cmd
cd /d C:\llama.cpp
```

Then run:

``` cmd
build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -cnv
```

Alternatively, from the models directory, the executable could be
referenced with:

``` cmd
..\build\bin\Release\llama-cli.exe -m "Qwen3-4B-Q4_K_M.gguf" -cnv
```

The first method is cleaner and easier to remember.

------------------------------------------------------------------------


# 21. First successful local inference

The model successfully responded to:

``` text
hi
```

The reported performance was approximately:

``` text
Prompt:     28.8 t/s
Generation: 10.2 t/s
```

This established a baseline.

The first complete working local inference chain was:

``` text
CMD
 |
 v
llama-cli.exe
 |
 v
Qwen3-4B-Q4_K_M.gguf
 |
 v
Local hardware
```

At this stage there was no VS Code integration yet.

------------------------------------------------------------------------


# 22. Why llama-server was required for Continue

`llama-cli` is designed for interactive command-line use.

Continue, however, needs a model endpoint that it can call
programmatically.

Instead of installing Ollama, a local server included with llama.cpp was
used:

``` text
llama-server.exe
```

This provides an HTTP API.

The architecture became:

``` text
Continue
 |
 v
http://127.0.0.1:8080
 |
 v
llama-server.exe
 |
 v
Qwen3 GGUF
```

This preserved the "no external middleman" requirement.

------------------------------------------------------------------------


# 23. Starting llama-server

From:

``` cmd
C:\llama.cpp>
```

the server was started with:

``` cmd
build\bin\Release\llama-server.exe -m "models\Qwen3-4B-Q4_K_M.gguf" --host 127.0.0.1 --port 8080
```

The important endpoint was:

``` text
http://127.0.0.1:8080
```

The server process must remain running while Continue uses the model.

------------------------------------------------------------------------


# 24. Problem: PowerShell curl behaved differently

When testing:

``` powershell
curl http://127.0.0.1:8080/v1/models
```

PowerShell interpreted `curl` as `Invoke-WebRequest`.

It displayed a security warning about parsing web content.

## Resolution

Use the real executable explicitly:

``` powershell
curl.exe http://127.0.0.1:8080/v1/models
```

This is an important Windows troubleshooting rule:

``` text
PowerShell:
curl      -> alias/function behavior may differ

PowerShell:
curl.exe  -> actual curl executable
```

------------------------------------------------------------------------


# 25. Confirming llama-server's model ID

The successful `/v1/models` response reported:

``` text
id:
models\Qwen3-4B-Q4_K_M.gguf
```

The server also reported useful model information:

``` text
n_vocab:       151936
n_ctx:          40960
n_embd:        2560
n_params:      4022468096
size:          2491323904
quantization:  Q4_K - Medium
```

The exact model ID matters when an external client such as Continue asks
llama-server which model to use.

------------------------------------------------------------------------


# 26. Installing Continue

Continue was installed as a VS Code extension.

The purpose was:

``` text
VS Code
 +
Continue
 +
local llama.cpp server
```

rather than a cloud AI provider.

------------------------------------------------------------------------


# 27. Configuring Continue

Continue's UI showed an API configuration screen.

The provider was changed from:

``` text
OpenRouter
```

to:

``` text
OpenAI Compatible
```

The important values were:

``` text
API Provider:
OpenAI Compatible

Base URL:
http://127.0.0.1:8080/v1
```

The API key was set to a local placeholder because the local llama.cpp
endpoint was not configured for authentication.

For example:

``` text
local
```

The model identifier must correspond to the model exposed by
llama-server.

The server had reported:

``` text
models\Qwen3-4B-Q4_K_M.gguf
```

Therefore that exact ID was used when required by the client.

Custom headers were not required.

Azure settings were left disabled.

------------------------------------------------------------------------


# 28. Problem: Cline caused configuration trouble

Cline was installed and initially investigated as an alternative coding
agent.

It required API configuration and produced several
workflow/configuration complications.

One issue involved:

``` text
Cannot use checkpoints in Desktop directory
```

This was a Cline workspace/checkpoint issue rather than a llama.cpp
problem.

The project could be moved to a normal development directory such as:

``` text
C:\Projects\
```

instead of directly working from the Desktop.

However, Cline ultimately became more trouble than necessary for this
setup.

## Decision

Cline was abandoned.

Continue was used instead.

This is an important practical lesson:

**Once the underlying local model/API is proven to work, do not
repeatedly rebuild the model stack because a VS Code extension is
troublesome. Replace the client layer.**

------------------------------------------------------------------------


# 29. Continue ultimately worked

After moving to Continue and configuring the local endpoint, Continue
successfully connected to the local llama.cpp infrastructure.

The final logical chain was:

``` text
VS Code
 |
 v
Continue
 |
 v
OpenAI-compatible local endpoint
 |
 v
127.0.0.1:8080
 |
 v
llama-server
 |
 v
Qwen3-4B-Q4_K_M.gguf
 |
 v
Local hardware
```

This was the successful endpoint of the setup.

------------------------------------------------------------------------


# 30. Final working architecture

The entire system can be visualized as:

``` text
                    ┌──────────────────────┐
                    │       VS CODE        │
                    └──────────┬───────────┘
                               │
                               v
                    ┌──────────────────────┐
                    │      CONTINUE        │
                    └──────────┬───────────┘
                               │
                               │ HTTP
                               │
                    ┌──────────v───────────┐
                    │   127.0.0.1:8080    │
                    └──────────┬───────────┘
                               │
                               v
                    ┌──────────────────────┐
                    │    llama-server      │
                    └──────────┬───────────┘
                               │
                               v
                    ┌──────────────────────┐
                    │ Qwen3-4B-Q4_K_M.gguf │
                    └──────────┬───────────┘
                               │
                               v
                    ┌──────────────────────┐
                    │   CPU / RAM / iGPU   │
                    └──────────────────────┘
```

------------------------------------------------------------------------


# 31. Complete troubleshooting table

  ----------------------------------------------------------------------------------------
  Problem                               Cause                   Solution
  ------------------------------------- ----------------------- --------------------------
  `git pull ... not a git repository`   Repository wasn't       Use `git clone`
                                        cloned                  

  `cmake not recognized`                CMake unavailable/PATH  Install CMake and reopen
                                        missing                 terminal

  `nmake failed`                        Compiler environment    Install/use MSVC Build
                                        unavailable             Tools

  `CMAKE_C_COMPILER not set`            C/C++ compiler not      Install Desktop
                                        available               development with C++

  `vswhere not recognized`              vswhere not on PATH     Don't depend on it; locate
                                                                Visual Studio manually

  `VsDevCmd.bat` not found              Wrong installation path Locate actual VS Build
                                        searched                Tools installation

  `/s /b` PowerShell error              CMD syntax used in      Use PowerShell
                                        PowerShell              `Get-ChildItem` syntax

  `cl` not recognized                   Developer environment   Run
                                        not loaded              `VsDevCmd.bat -arch=x64`

  CMake reused bad configuration        Old `build` directory   `rmdir /s /q build` and
                                        retained                reconfigure

  Many model-related files during build llama.cpp compiles      Normal; no model weights
                                        architecture support    are being downloaded

  `npm ci` during build                 llama.cpp UI assets     Normal during complete
                                                                build

  CLI path not found                    Wrong working directory `cd /d C:\llama.cpp`

  `curl` security warning in PowerShell PowerShell aliases      Use `curl.exe`
                                        `curl`                  

  Cline not responding                  Client/configuration    Test `/v1/models` and
                                        issue                   generation independently

  Cline checkpoint error on Desktop     Workspace/checkpoint    Move project under
                                        restriction             `C:\Projects`

  Cline too complicated                 Agent/client overhead   Use Continue

  Continue configuration confusion      Provider/model endpoint Use local llama.cpp
                                        mismatch                endpoint and exact model
                                                                ID
  ----------------------------------------------------------------------------------------

------------------------------------------------------------------------


# 32. Most important troubleshooting methodology

The biggest lesson from the setup is to debug the system in layers.

Do not debug everything simultaneously.

Use this order:

## Layer 1 --- Compiler

``` cmd
cl
```

If this fails, don't touch llama.cpp.

## Layer 2 --- CMake

``` cmd
cmake --version
```

Then:

``` cmd
cmake -B build
```

## Layer 3 --- llama.cpp binary

``` cmd
build\bin\Release\llama-cli.exe --version
```

## Layer 4 --- Model

``` cmd
build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -cnv
```

If this works, model loading is good.

## Layer 5 --- Server

Start:

``` cmd
build\bin\Release\llama-server.exe -m "models\Qwen3-4B-Q4_K_M.gguf" --host 127.0.0.1 --port 8080
```

Then:

``` powershell
curl.exe http://127.0.0.1:8080/v1/models
```

## Layer 6 --- Client

Only now configure:

``` text
Continue
```

This approach prevents a VS Code extension problem from being mistaken
for a model problem.

------------------------------------------------------------------------


# 33. Daily usage

For normal use, you do not need to rebuild anything.

### Terminal 1 --- start llama-server

``` cmd
cd /d C:\llama.cpp
build\bin\Release\llama-server.exe -m "models\Qwen3-4B-Q4_K_M.gguf" --host 127.0.0.1 --port 8080
```

Leave it running.

### Terminal 2 / VS Code

Open VS Code and use Continue.

The model is already loaded by llama-server.

------------------------------------------------------------------------


# 34. Updating llama.cpp later

When you want to update the llama.cpp source:

``` cmd
cd /d C:\llama.cpp
git pull
```

Then rebuild:

``` cmd
cmake --build build --config Release --parallel
```

If a major build/configuration problem occurs, start clean:

``` cmd
rmdir /s /q build
cmake -B build -G "Visual Studio 18 2026"
cmake --build build --config Release --parallel
```

Do not delete your `models` directory unless you intentionally want to
remove the model.

------------------------------------------------------------------------


# 35. Model management

Your model files are independent from the llama.cpp installation.

For example:

``` text
C:\llama.cpp\models\
    Qwen3-4B-Q4_K_M.gguf
```

You can later add:

``` text
C:\llama.cpp\models\
    Qwen3-4B-Q4_K_M.gguf
    another-model.gguf
    coding-model.gguf
```

Then choose the model when launching llama-server.

------------------------------------------------------------------------


# 36. Security and privacy

The server was bound to:

``` text
127.0.0.1
```

This means the endpoint is intended to be accessible only from the local
machine.

The important distinction is:

``` text
127.0.0.1
```

versus:

``` text
0.0.0.0
```

For a private local coding assistant, keeping:

``` text
--host 127.0.0.1
```

is preferable unless remote access is explicitly required.

Do not expose the server to your LAN/Internet without understanding
authentication, firewall, and network-security implications.

------------------------------------------------------------------------


# 37. Performance baseline

The first direct CLI run produced approximately:

``` text
Prompt processing: 28.8 tokens/sec
Generation:        10.2 tokens/sec
```

This is the baseline to preserve.

Future optimizations can be evaluated against it.

For example:

``` text
CPU baseline:
10.2 t/s

Vulkan:
? t/s

Different quantization:
? t/s

Different model:
? t/s
```

Always benchmark rather than assuming GPU acceleration will be faster on
an integrated GPU.

------------------------------------------------------------------------


# 38. Next possible improvement: Intel Vulkan

The Intel Iris Xe GPU was not yet fully optimized in the final working
setup.

A logical next experiment is to build llama.cpp with Vulkan support.

The intended architecture would become:

``` text
Continue
   |
   v
llama-server
   |
   v
llama.cpp
   |
   v
Vulkan
   |
   v
Intel Iris Xe
```

However, this should be treated as a separate optimization experiment.

Do not destroy the current CPU build.

Keep the working configuration as a baseline.

------------------------------------------------------------------------


# 39. Next possible improvement: coding-specific model

Qwen3-4B is a useful general model, but a 4B model is not equivalent to
a large frontier coding model.

For serious coding-agent workloads, limitations may appear in:

-   multi-file reasoning;
-   long context;
-   complex refactoring;
-   tool use;
-   debugging;
-   architecture planning;
-   maintaining consistency across many files.

A future upgrade should evaluate a coding-oriented model that fits the
available RAM.

------------------------------------------------------------------------


# 40. Why a local coding agent may feel slow

There are several independent sources of latency:

``` text
Model loading
+
Prompt processing
+
Context size
+
Generation speed
+
Tool calls
+
File reading
+
Large repository context
```

A coding agent can be much slower than a simple chatbot because it may
repeatedly:

``` text
read file
→ reason
→ call tool
→ inspect result
→ reason
→ modify file
→ inspect errors
→ reason again
```

Therefore, local 4B models are best introduced gradually.

------------------------------------------------------------------------


# 41. Recommended final directory structure

Keep the installation organized like this:

``` text
C:\
│
├── llama.cpp\
│   │
│   ├── build\
│   │   └── bin\
│   │       └── Release\
│   │           ├── llama-cli.exe
│   │           ├── llama-server.exe
│   │           └── ...
│   │
│   ├── models\
│   │   └── Qwen3-4B-Q4_K_M.gguf
│   │
│   └── ...
│
└── Projects\
    ├── project-a\
    ├── project-b\
    └── project-c\
```

This avoids mixing:

``` text
model files
source code
build files
VS Code projects
```

------------------------------------------------------------------------


# 42. Quick recovery checklist

If the system stops working, use this sequence.

### A. Is the executable present?

``` cmd
dir C:\llama.cpp\build\bin\Release\llama-cli.exe
```

### B. Is the model present?

``` cmd
dir C:\llama.cpp\models\Qwen3-4B-Q4_K_M.gguf
```

### C. Does llama.cpp run?

``` cmd
C:\llama.cpp\build\bin\Release\llama-cli.exe --version
```

### D. Can the model load?

``` cmd
cd /d C:\llama.cpp
build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -cnv
```

### E. Is the server running?

``` cmd
build\bin\Release\llama-server.exe -m "models\Qwen3-4B-Q4_K_M.gguf" --host 127.0.0.1 --port 8080
```

### F. Is the API reachable?

PowerShell:

``` powershell
curl.exe http://127.0.0.1:8080/v1/models
```

### G. Is Continue configured to use the local endpoint?

Check:

``` text
Provider:
local/OpenAI-compatible/llama.cpp configuration

Endpoint:
http://127.0.0.1:8080/v1

Model:
the exact model ID returned by /v1/models
```

------------------------------------------------------------------------


# 43. Core lessons from the entire setup

### Lesson 1

`git clone` is for the initial repository download.

### Lesson 2

CMake is not a compiler.

### Lesson 3

VS Code is not a compiler.

### Lesson 4

Windows development environments are sensitive to CMD vs PowerShell.

### Lesson 5

A CMake build directory can retain a bad configuration. Delete `build`
and reconfigure when necessary.

### Lesson 6

A model file and the inference engine are separate things.

### Lesson 7

Many model-related source files during a llama.cpp build do not mean
many models are being downloaded.

### Lesson 8

`llama-cli` is sufficient for direct terminal inference.

### Lesson 9

`llama-server` is useful when another application needs to communicate
with llama.cpp.

### Lesson 10

Test the inference API before debugging a VS Code extension.

### Lesson 11

Use the exact model ID returned by `/v1/models` when a client requires
it.

### Lesson 12

Keep a working baseline before experimenting with GPU acceleration or
larger models.

------------------------------------------------------------------------


# 44. Final result

The completed system is:

``` text
                     LOCAL AI CODING STACK

                         VS CODE
                            |
                            v
                        CONTINUE
                            |
                            | localhost
                            v
                 http://127.0.0.1:8080/v1
                            |
                            v
                    llama-server.exe
                            |
                            v
                 Qwen3-4B-Q4_K_M.gguf
                            |
                            v
                  Local Windows Hardware
                  /                    \
                 CPU              Intel Iris Xe
                            +
                         ~16 GB RAM
```

The entire inference path is local.

The setup does not depend on:

``` text
Ollama
OpenRouter
OpenAI API
Anthropic API
Gemini API
Cloud GPU
Hosted inference provider
```

The only external download involved in the model stage was the GGUF
model file itself, and the software source was cloned from the llama.cpp
repository.

------------------------------------------------------------------------


# 45. Minimal commands to remember — backend-aware

## Windows x64 CPU


In [ ]:
!cd /d C:\llama.cpp
!build\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -cnv


## Windows x64 CPU server


In [ ]:
!cd /d C:\llama.cpp
!build\bin\Release\llama-server.exe -m "models\Qwen3-4B-Q4_K_M.gguf" --host 127.0.0.1 --port 8080


## Test local API

PowerShell:


In [ ]:
!curl.exe http://127.0.0.1:8080/v1/models


## Vulkan build


In [ ]:
!cd /d C:\llama.cpp
!cmake -B build-vulkan -DGGML_VULKAN=ON
!cmake --build build-vulkan --config Release -j


Run:


In [ ]:
!build-vulkan\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -ngl 99 -cnv


## CUDA build


In [ ]:
!cd /d C:\llama.cpp
!cmake -B build-cuda -DGGML_CUDA=ON
!cmake --build build-cuda --config Release -j


Run:


In [ ]:
!build-cuda\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -ngl 99 -cnv


## Intel SYCL build

After installing Intel oneAPI:


In [ ]:
!call "C:\Program Files (x86)\Intel\oneAPI\setvars.bat" intel64 --force
!cd /d C:\llama.cpp
!cmake -B build-sycl -G "Ninja" -DGGML_SYCL=ON -DCMAKE_C_COMPILER=cl -DCMAKE_CXX_COMPILER=icx -DCMAKE_BUILD_TYPE=Release
!cmake --build build-sycl --config Release -j


## Update source


In [ ]:
!cd /d C:\llama.cpp
!git pull


## Rebuild CPU version


In [ ]:
!cmake --build build --config Release -j


## Clean CPU reconfiguration


In [ ]:
!rmdir /s /q build
!cmake -B build
!cmake --build build --config Release -j


## Verify binaries


In [ ]:
!build\bin\Release\llama-cli.exe --version
!build\bin\Release\llama-server.exe --help

# 46. Official links and references

The links below are the primary references used for this setup. Prefer these official sources over third-party tutorials when reinstalling or troubleshooting.

## Core software

### llama.cpp — official GitHub repository
[https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)

Main source repository for llama.cpp, including `llama-cli`, `llama-server`, build instructions, supported backends, releases, and API documentation.

### llama.cpp — official build documentation
[https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)

Use this when rebuilding llama.cpp, changing GPU backends, or troubleshooting CMake/build configuration.

### llama.cpp — official releases
[https://github.com/ggml-org/llama.cpp/releases](https://github.com/ggml-org/llama.cpp/releases)

Useful if you want to use prebuilt binaries instead of compiling from source.

### llama.cpp — official VS Code/FIM extension
[https://github.com/ggml-org/llama.vscode](https://github.com/ggml-org/llama.vscode)

Official llama.cpp-related VS Code extension for fill-in-the-middle/code completion workflows.

---

## Build tools

### CMake — official download
[https://cmake.org/download/](https://cmake.org/download/)

Used to configure and generate the llama.cpp build.

### Visual Studio Downloads — official Microsoft page
[https://visualstudio.microsoft.com/downloads/](https://visualstudio.microsoft.com/downloads/)

Contains the current Visual Studio IDE downloads and **Build Tools for Visual Studio**.

### Visual Studio Build Tools
[https://visualstudio.microsoft.com/downloads/](https://visualstudio.microsoft.com/downloads/)

On the Microsoft downloads page, go to:

```text
Tools for Visual Studio
    ↓
Build Tools for Visual Studio
```

For this project, select the C++ workload:

```text
Desktop development with C++
```

The Microsoft downloads page currently lists Build Tools for Visual Studio 2026 as the command-line build package for C++ desktop projects. 

### Visual Studio C++ command-line build documentation
[https://learn.microsoft.com/cpp/build/building-on-the-command-line](https://learn.microsoft.com/cpp/build/building-on-the-command-line)

Useful for understanding `cl`, developer command prompts, MSVC environments, and command-line C++ builds.

---

## Git

### Git — official website
[https://git-scm.com/](https://git-scm.com/)

### Git documentation
[https://git-scm.com/doc](https://git-scm.com/doc)

Basic commands used in this setup:


In [ ]:
!git clone https://github.com/ggml-org/llama.cpp.git


and later:


In [ ]:
!git pull


---

## VS Code

### Visual Studio Code — official website
[https://code.visualstudio.com/](https://code.visualstudio.com/)

### Visual Studio Marketplace
[https://marketplace.visualstudio.com/](https://marketplace.visualstudio.com/)

Use the Marketplace to install the Continue extension.

---

## Continue

### Continue — official website
[https://www.continue.dev/](https://www.continue.dev/)

### Continue — official documentation
[https://docs.continue.dev/](https://docs.continue.dev/)

### Continue — install documentation
[https://docs.continue.dev/ide-extensions/install](https://docs.continue.dev/ide-extensions/install)

This explains how to install Continue in VS Code and other supported IDEs.

### Continue — llama.cpp provider documentation
[https://docs.continue.dev/customize/model-providers/more/llamacpp](https://docs.continue.dev/customize/model-providers/more/llamacpp)

This is the most relevant Continue page for this setup. It documents the local llama.cpp provider and the `apiBase` configuration.

The documented pattern is:

```yaml
models:
  - name: <MODEL_NAME>
    provider: llama.cpp
    model: <MODEL_ID>
    apiBase: http://localhost:8080
```

### Continue — configuration reference
[https://docs.continue.dev/reference](https://docs.continue.dev/reference)

Useful when adding models, rules, tools, context providers, and other Continue configuration.

---

## Qwen3 model

### Qwen3-4B-GGUF — official Qwen Hugging Face repository
[https://huggingface.co/Qwen/Qwen3-4B-GGUF](https://huggingface.co/Qwen/Qwen3-4B-GGUF)

This is the model repository used for the Qwen3-4B GGUF files.

### Exact Q4_K_M model file
[https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf](https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf)

This is the exact model variant used in this setup.

### Direct download URL used in CMD

```text
https://huggingface.co/Qwen/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf?download=true
```

CMD command:


In [ ]:
!curl -L -o Qwen3-4B-Q4_K_M.gguf "https://huggingface.co/Qwen/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf?download=true"


### Qwen3 model documentation/model card
[https://huggingface.co/Qwen/Qwen3-4B-GGUF](https://huggingface.co/Qwen/Qwen3-4B-GGUF)

The model card documents the available quantizations, context information, license, and llama.cpp usage.

The Qwen3-4B GGUF repository lists Q4_K_M alongside other quantizations such as Q5_K_M, Q6_K, and Q8_0. 

---

## Hugging Face

### Hugging Face — official website
[https://huggingface.co/](https://huggingface.co/)

Hugging Face hosts the GGUF model repository used in this guide.

### Qwen organization
[https://huggingface.co/Qwen](https://huggingface.co/Qwen)

Useful for finding other Qwen model families and official GGUF releases.

---

## Local API testing

### llama.cpp server endpoint used in this setup

```text
http://127.0.0.1:8080
```

Model listing endpoint:

```text
http://127.0.0.1:8080/v1/models
```

OpenAI-compatible chat endpoint:

```text
http://127.0.0.1:8080/v1/chat/completions
```

PowerShell test:


In [ ]:
!curl.exe http://127.0.0.1:8080/v1/models


---

## Intel GPU / future optimization

### llama.cpp Vulkan backend documentation
[https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)

The official llama.cpp project supports Vulkan and SYCL backends in addition to CPU execution and NVIDIA CUDA. 

For the Intel Iris Xe machine used in this guide, Vulkan is the next backend worth investigating.

Do not replace the known-good CPU build until a Vulkan build has been successfully tested.

---

## Useful source-code/API references

### llama.cpp GitHub repository
[https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)

### llama.cpp server/API documentation
[https://github.com/ggml-org/llama.cpp/tree/master/tools/server](https://github.com/ggml-org/llama.cpp/tree/master/tools/server)

### llama.cpp examples
[https://github.com/ggml-org/llama.cpp/tree/master/examples](https://github.com/ggml-org/llama.cpp/tree/master/examples)

---


# 47. Recommended reference order

When troubleshooting this installation, use sources in this order:

1. **llama.cpp GitHub**
   [https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)

2. **llama.cpp build documentation**
   [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)

3. **Continue llama.cpp provider documentation**
   [https://docs.continue.dev/customize/model-providers/more/llamacpp](https://docs.continue.dev/customize/model-providers/more/llamacpp)

4. **Qwen3-4B GGUF model card**
   [https://huggingface.co/Qwen/Qwen3-4B-GGUF](https://huggingface.co/Qwen/Qwen3-4B-GGUF)

5. **Microsoft Visual Studio Build Tools**
   [https://visualstudio.microsoft.com/downloads/](https://visualstudio.microsoft.com/downloads/)

6. **CMake**
   [https://cmake.org/download/](https://cmake.org/download/)

7. **VS Code**
   [https://code.visualstudio.com/](https://code.visualstudio.com/)

This order keeps troubleshooting close to the actual software maintainers.

---


# 48. Quick link sheet

| Purpose | Link |
|---|---|
| llama.cpp source | [https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp) |
| llama.cpp build guide | [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md) |
| llama.cpp releases | [https://github.com/ggml-org/llama.cpp/releases](https://github.com/ggml-org/llama.cpp/releases) |
| llama.cpp VS Code/FIM | [https://github.com/ggml-org/llama.vscode](https://github.com/ggml-org/llama.vscode) |
| CMake | [https://cmake.org/download/](https://cmake.org/download/) |
| Visual Studio Build Tools | [https://visualstudio.microsoft.com/downloads/](https://visualstudio.microsoft.com/downloads/) |
| Microsoft C++ CLI docs | [https://learn.microsoft.com/cpp/build/building-on-the-command-line](https://learn.microsoft.com/cpp/build/building-on-the-command-line) |
| Git | [https://git-scm.com/](https://git-scm.com/) |
| VS Code | [https://code.visualstudio.com/](https://code.visualstudio.com/) |
| VS Code Marketplace | [https://marketplace.visualstudio.com/](https://marketplace.visualstudio.com/) |
| Continue | [https://www.continue.dev/](https://www.continue.dev/) |
| Continue installation | [https://docs.continue.dev/ide-extensions/install](https://docs.continue.dev/ide-extensions/install) |
| Continue llama.cpp provider | [https://docs.continue.dev/customize/model-providers/more/llamacpp](https://docs.continue.dev/customize/model-providers/more/llamacpp) |
| Continue configuration reference | [https://docs.continue.dev/reference](https://docs.continue.dev/reference) |
| Qwen3-4B GGUF | [https://huggingface.co/Qwen/Qwen3-4B-GGUF](https://huggingface.co/Qwen/Qwen3-4B-GGUF) |
| Exact Q4_K_M file | [https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf](https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf) |
| Hugging Face | [https://huggingface.co/](https://huggingface.co/) |
| Qwen on Hugging Face | [https://huggingface.co/Qwen](https://huggingface.co/Qwen) |

---


# 46. Learning resources mapped to each setup stage

## Git — clone vs pull
- Git documentation: [https://git-scm.com/doc](https://git-scm.com/doc)
- `git clone`: [https://git-scm.com/docs/git-clone](https://git-scm.com/docs/git-clone)
- `git pull`: [https://git-scm.com/docs/git-pull](https://git-scm.com/docs/git-pull)

## CMake
- CMake downloads: [https://cmake.org/download/](https://cmake.org/download/)
- CMake documentation: [https://cmake.org/documentation/](https://cmake.org/documentation/)
- CMake CLI reference: [https://cmake.org/cmake/help/latest/manual/cmake.1.html](https://cmake.org/cmake/help/latest/manual/cmake.1.html)
- Video: [https://www.youtube.com/watch?v=L4yNhSX2ihs](https://www.youtube.com/watch?v=L4yNhSX2ihs)
  - Windows llama.cpp build walkthrough covering Git, Visual Studio Build Tools, CMake, Developer Command Prompt and compilation.
  - **Caution:** its GPU portion is CUDA/NVIDIA-specific.

## Visual Studio Build Tools / MSVC
- Visual Studio downloads: [https://visualstudio.microsoft.com/downloads/](https://visualstudio.microsoft.com/downloads/)
- Visual Studio installation: [https://learn.microsoft.com/visualstudio/install/install-visual-studio](https://learn.microsoft.com/visualstudio/install/install-visual-studio)
- Modify workloads/components: [https://learn.microsoft.com/visualstudio/install/modify-visual-studio](https://learn.microsoft.com/visualstudio/install/modify-visual-studio)
- Developer Command Prompt: [https://learn.microsoft.com/visualstudio/ide/reference/command-prompt-powershell](https://learn.microsoft.com/visualstudio/ide/reference/command-prompt-powershell)
- Microsoft C++ docs: [https://learn.microsoft.com/cpp/](https://learn.microsoft.com/cpp/)
- llama.cpp Windows build guide: [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)
- Video: [https://www.youtube.com/watch?v=L4yNhSX2ihs](https://www.youtube.com/watch?v=L4yNhSX2ihs)

## CMD vs PowerShell
- PowerShell: [https://learn.microsoft.com/powershell/](https://learn.microsoft.com/powershell/)
- `Get-ChildItem`: [https://learn.microsoft.com/powershell/module/microsoft.powershell.management/get-childitem](https://learn.microsoft.com/powershell/module/microsoft.powershell.management/get-childitem)
- Windows command reference: [https://learn.microsoft.com/windows-server/administration/windows-commands/windows-commands](https://learn.microsoft.com/windows-server/administration/windows-commands/windows-commands)

## Building llama.cpp
- Official repository: [https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)
- Official build guide: [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)
- Older Windows/CMake video: [https://www.youtube.com/watch?v=coIj2CU5LMU](https://www.youtube.com/watch?v=coIj2CU5LMU)
- **Caution:** this video is from 2023; use current official docs for commands.

## GGUF / Qwen3 model
- Qwen organization: [https://huggingface.co/Qwen](https://huggingface.co/Qwen)
- Qwen3-4B GGUF: [https://huggingface.co/Qwen/Qwen3-4B-GGUF](https://huggingface.co/Qwen/Qwen3-4B-GGUF)
- Exact Q4_K_M file: [https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf](https://huggingface.co/Qwen/Qwen3-4B-GGUF/blob/main/Qwen3-4B-Q4_K_M.gguf)
- Hugging Face docs: [https://huggingface.co/docs/hub/](https://huggingface.co/docs/hub/)
- Direct download used:
  [https://huggingface.co/Qwen/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf?download=true](https://huggingface.co/Qwen/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf?download=true)

## llama-cli
- llama.cpp documentation: [https://github.com/ggml-org/llama.cpp/tree/master/docs](https://github.com/ggml-org/llama.cpp/tree/master/docs)
- Windows local-install video: [https://www.youtube.com/watch?v=wZ8C40TNE4Q](https://www.youtube.com/watch?v=wZ8C40TNE4Q)

## llama-server
- Official server README: [https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md)
- Server documentation: [https://www.mintlify.com/ggml-org/llama.cpp/inference/server](https://www.mintlify.com/ggml-org/llama.cpp/inference/server)
- Windows WebUI/server video: [https://www.youtube.com/watch?v=wZ8C40TNE4Q](https://www.youtube.com/watch?v=wZ8C40TNE4Q)

The official server README documents `/v1/models` and `/v1/chat/completions`, and explains that the default model ID is the path supplied to `-m` unless `--alias` is used. citeturn0search0

## Testing the API
- curl documentation: [https://curl.se/docs/](https://curl.se/docs/)
- PowerShell: [https://learn.microsoft.com/powershell/](https://learn.microsoft.com/powershell/)
- Test command:

In [ ]:
!  curl.exe http://127.0.0.1:8080/v1/models


## Continue
- Continue: [https://www.continue.dev/](https://www.continue.dev/)
- Continue docs: [https://docs.continue.dev/](https://docs.continue.dev/)
- Installation: [https://docs.continue.dev/ide-extensions/install](https://docs.continue.dev/ide-extensions/install)
- llama.cpp provider: [https://docs.continue.dev/customize/model-providers/more/llamacpp](https://docs.continue.dev/customize/model-providers/more/llamacpp)
- Provider overview: [https://docs.continue.dev/customize/model-providers/overview](https://docs.continue.dev/customize/model-providers/overview)
- Configuration reference: [https://docs.continue.dev/reference](https://docs.continue.dev/reference)
- Video: [https://www.youtube.com/watch?v=AV_8czoF3PU](https://www.youtube.com/watch?v=AV_8czoF3PU)
  - **Caution:** this video uses Ollama. Use it only for learning the Continue UI/workflow; this guide uses llama.cpp directly.

## Cline troubleshooting
- Cline: [https://cline.bot/](https://cline.bot/)
- Continue: [https://www.continue.dev/](https://www.continue.dev/)
- Principle: prove `llama-cli`, then `llama-server`, then the HTTP API, and only then debug the VS Code client.

## Updating llama.cpp
- Build guide: [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)
- Releases: [https://github.com/ggml-org/llama.cpp/releases](https://github.com/ggml-org/llama.cpp/releases)
- Git pull: [https://git-scm.com/docs/git-pull](https://git-scm.com/docs/git-pull)

## Intel Iris Xe / future acceleration
- llama.cpp build guide: [https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)
- Backend docs: [https://github.com/ggml-org/llama.cpp/tree/master/docs/backend](https://github.com/ggml-org/llama.cpp/tree/master/docs/backend)
- SYCL backend: [https://github.com/ggml-org/llama.cpp/blob/master/docs/backend/SYCL.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/backend/SYCL.md)
- GPU build video: [https://www.youtube.com/watch?v=L4yNhSX2ihs](https://www.youtube.com/watch?v=L4yNhSX2ihs)
  - **Caution:** CUDA sections are NVIDIA-specific and do not directly apply to Intel Iris Xe.

---


# 47. Video library

1. **Complete Llama.cpp Build Guide 2025 — Windows + GPU Acceleration**  
   [https://www.youtube.com/watch?v=L4yNhSX2ihs](https://www.youtube.com/watch?v=L4yNhSX2ihs)  
   Best for Windows build flow, Visual Studio Build Tools, CMake, Developer Command Prompt and compilation.

2. **llama.cpp Windows (CMake)**  
   [https://www.youtube.com/watch?v=coIj2CU5LMU](https://www.youtube.com/watch?v=coIj2CU5LMU)  
   Best for understanding the Windows/CMake concept. Older video; verify commands against current docs.

3. **llama.cpp OFFICIAL WebUI — Windows 11 Install Guide**  
   [https://www.youtube.com/watch?v=wZ8C40TNE4Q](https://www.youtube.com/watch?v=wZ8C40TNE4Q)  
   Best for seeing llama.cpp running locally on Windows and understanding the server/WebUI side.

4. **Local AI Coding in VS Code with Continue**  
   [https://www.youtube.com/watch?v=AV_8czoF3PU](https://www.youtube.com/watch?v=AV_8czoF3PU)  
   Best for understanding Continue's VS Code workflow. It uses Ollama, so do not copy the backend setup.

---


# 48. Resource priority

When a video and documentation disagree, use:

```text
1. Current official documentation
2. Current official GitHub README/docs
3. Current official model card
4. Recent technical video
5. Older tutorial
6. Community post
```

This is especially important for llama.cpp because build options and backends change frequently.

The current official build documentation covers Windows/MSVC, the Desktop development with C++ workload, CMake tools, and Developer Command Prompt/PowerShell usage. citeturn0search1

---


# 49. Recommended learning sequence

```text
Git
 ↓
CMake
 ↓
MSVC / Developer Command Prompt
 ↓
llama.cpp build
 ↓
GGUF + quantization
 ↓
llama-cli
 ↓
llama-server
 ↓
OpenAI-compatible HTTP API
 ↓
Continue
 ↓
GPU backends
 ↓
Model selection
 ↓
Coding-agent workflows
```


---


# 51. Command matrix by hardware

This is the most important correction for readers using hardware other than the original machine.

| Hardware | Configure | Build | Run |
|---|---|---|---|
| x86-64 CPU | `cmake -B build` | `cmake --build build --config Release -j` | `build\bin\Release\llama-cli.exe ...` |
| x86-64 portable CPU | `cmake -B build -DGGML_NATIVE=OFF` | `cmake --build build --config Release -j` | Same CPU binary |
| NVIDIA GPU | `cmake -B build-cuda -DGGML_CUDA=ON` | `cmake --build build-cuda --config Release -j` | `build-cuda\bin\Release\llama-cli.exe ... -ngl 99` |
| Vulkan GPU | `cmake -B build-vulkan -DGGML_VULKAN=ON` | `cmake --build build-vulkan --config Release -j` | `build-vulkan\bin\Release\llama-cli.exe ... -ngl 99` |
| Intel GPU / SYCL | oneAPI + `GGML_SYCL=ON` | Ninja build | SYCL binary + GPU offload |
| Windows ARM64 | official ARM64 preset | ARM64 preset build | ARM64 binary |
| Apple Silicon | `GGML_METAL=ON` | Metal build | macOS binary |
| Linux CPU | `cmake -B build` | `cmake --build build --config Release -j` | Linux binary |

---


# 52. Important command correction: don't hard-code the original CPU

The original successful command:


In [ ]:
!cmake -B build


remains correct for a normal CPU build.

What changes is the **backend**, not the fundamental llama.cpp workflow:

```text
CPU
  → cmake -B build

NVIDIA
  → cmake -B build-cuda -DGGML_CUDA=ON

Vulkan
  → cmake -B build-vulkan -DGGML_VULKAN=ON

Intel SYCL
  → cmake -B build-sycl -DGGML_SYCL=ON ...

ARM64
  → ARM64 preset

Apple
  → Metal
```

This is preferable to writing one command that tries to guess every user's hardware.

---


# 53. Preserve separate builds when experimenting

If someone wants to compare CPU and GPU performance, use separate build directories:


In [ ]:
!cmake -B build-cpu
!cmake -B build-vulkan -DGGML_VULKAN=ON
!cmake -B build-cuda -DGGML_CUDA=ON


Then:


In [ ]:
!cmake --build build-cpu --config Release -j
!cmake --build build-vulkan --config Release -j
!cmake --build build-cuda --config Release -j


This avoids accidentally replacing a known-good configuration.

The same model can then be tested against all three binaries.

---


# 54. Backend verification

Do not assume that a build is actually using the intended accelerator.

For CPU:


In [ ]:
!build\bin\Release\llama-cli.exe --version


For Vulkan:


In [ ]:
!build-vulkan\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -ngl 99 -cnv


Look at startup logs for a Vulkan GPU/backend being detected.

For SYCL:


In [ ]:
!build-sycl\bin\Release\llama-cli.exe --help


Then run a model and inspect startup logs for SYCL/Intel device detection.

For CUDA:


In [ ]:
!build-cuda\bin\Release\llama-cli.exe -m "models\Qwen3-4B-Q4_K_M.gguf" -ngl 99 -cnv


Inspect startup output for CUDA device detection.

The principle is:

```text
Build flag ≠ proof of runtime acceleration
```

The runtime log is the final verification.

---


# 55. Hardware-neutral troubleshooting rule

When another user reports that "llama.cpp is slow", first collect:

```text
CPU model
CPU instruction support
RAM capacity
RAM speed/channel configuration
GPU model
GPU VRAM
OS
llama.cpp commit/version
GGUF model
quantization
context size
backend
GPU layers (-ngl)
threads
```

Do not prescribe a new model or build flag before collecting these values.

This prevents a CPU-only solution from being incorrectly applied to a CUDA/Vulkan/SYCL system, or vice versa.


---


# 56. GPU-first hardware selection and model sizing

CPU-only inference is only one part of the picture. For local LLMs, **GPU VRAM is often the most important hardware constraint once a suitable accelerator is available**.

The correct decision process is:

```text
CPU
+
System RAM
+
GPU
+
GPU VRAM
+
GPU backend
+
Model quantization
+
Context length
=
Actual usable configuration
```

## 56.1 GPU decision table

| GPU class | Examples | Main backend | Good starting model range | Main limitation |
|---|---|---|---|---|
| Integrated graphics | Intel Iris Xe, Radeon iGPU | CPU / Vulkan / SYCL where supported | 1B–8B | Shared system memory |
| Entry NVIDIA | GTX 1650/1660, RTX 3050-class | CUDA | 3B–8B | VRAM |
| RTX 3060 12 GB | RTX 3060 12 GB | CUDA | 7B–14B Q4 | 12 GB VRAM |
| RTX 4060/5060-class 8 GB | 8 GB VRAM | CUDA | 3B–8B Q4 | 8 GB VRAM |
| RTX 4070-class 12 GB | 12 GB VRAM | CUDA | 7B–14B Q4 | VRAM |
| RTX 4080-class 16 GB | 16 GB VRAM | CUDA | 7B–20B Q4 | VRAM |
| RTX 4090-class 24 GB | 24 GB VRAM | CUDA | 14B–32B Q4/Q5 | VRAM |
| RTX 5090-class 32 GB | 32 GB VRAM | CUDA | 14B–32B+ Q4/Q5 | VRAM |
| AMD 16 GB+ GPU | Radeon/Pro variants | ROCm/HIP or Vulkan | 7B–32B depending on VRAM | Backend compatibility |
| Intel Arc 8–16+ GB | Arc variants | Vulkan / SYCL | 7B–20B+ depending on VRAM | Backend/software maturity |
| Apple Silicon 16–128 GB unified memory | M-series | Metal | RAM-dependent | Unified-memory allocation |

These are **starting points**, not guarantees. Model architecture, quantization, context length, GPU generation and backend implementation can materially change performance.

## 56.2 VRAM is more important than GPU marketing tier

For LLM inference, do not evaluate a GPU only by:

```text
RTX 4090 > RTX 4080 > RTX 4070
```

Instead ask:

```text
How much VRAM?
What memory bandwidth?
Which backend?
Can the model fit?
How much of the model must remain in system RAM?
```

A GPU with more VRAM can sometimes be a substantially better local-LLM purchase than a faster GPU with less VRAM.

## 56.3 Approximate model-weight sizing

A rough Q4 estimate is:

```text
Model parameters × ~0.5 bytes
```

So, very approximately:

| Model | Q4 weights, rough starting estimate |
|---:|---:|
| 1B | ~0.5–0.8 GB |
| 3B | ~1.5–2.0 GB |
| 4B | ~2–3 GB |
| 7B | ~4–5 GB |
| 8B | ~5 GB |
| 14B | ~8–10 GB |
| 20B | ~11–13 GB |
| 32B | ~18–20 GB |
| 70B | ~38–45 GB |

**These are not total VRAM requirements.**

You must also account for:

```text
GGUF weights
+
KV cache
+
compute buffers
+
CUDA/Vulkan/SYCL runtime allocations
+
context length
```

Therefore, do not buy a GPU with exactly the same VRAM as the model's approximate file size.

## 56.4 Practical VRAM planning

Use this as a conservative rule:

```text
4 GB VRAM
 → 1B–3B Q4

6 GB VRAM
 → 3B–7B Q4

8 GB VRAM
 → 3B–8B Q4

12 GB VRAM
 → 7B–14B Q4

16 GB VRAM
 → 7B–20B Q4

24 GB VRAM
 → 14B–32B Q4/Q5

32 GB VRAM
 → 20B–32B+ Q4/Q5

48 GB VRAM
 → 32B–70B depending on quantization/context

80 GB+
 → large 70B-class and larger workloads
```

Treat these as **safe planning ranges**, not hard model limits.

## 56.5 Context length changes memory requirements

A common mistake is:

```text
"The 8B Q4 model is 5 GB, so 6 GB VRAM is enough."
```

Not necessarily.

Increasing:

```text
context = 8K
```

to:

```text
context = 32K
```

can substantially increase KV-cache memory.

Therefore:

```text
Model size alone
        ≠
Total inference memory
```

For coding assistants, context can become particularly important because the client may send:

```text
current file
+
related files
+
repository context
+
system prompt
+
tool output
+
conversation history
```

## 56.6 Full GPU offload vs partial offload

llama.cpp can split work between GPU and CPU.

Example:


In [ ]:
!llama-cli.exe -m "models\model.gguf" -ngl 99 -cnv


`-ngl 99` requests a high number of layers to be offloaded.

If the model does not fit:

```text
GPU VRAM insufficient
        ↓
Reduce GPU layers
        ↓
Some layers remain on CPU
```

This means a model does **not necessarily have to fit entirely inside VRAM**.

However, partial offloading can be substantially slower than keeping the model and relevant workload on the GPU because data must move between CPU/system memory and GPU memory.

## 56.7 Multiple GPUs

For systems with multiple GPUs, evaluate:

```text
GPU 1 VRAM
+
GPU 2 VRAM
+
interconnect
+
PCIe topology
+
backend support
```

Do not simply assume:

```text
12 GB + 12 GB = identical to a 24 GB GPU
```

Memory bandwidth and communication overhead matter.

For multi-GPU configurations, use the current llama.cpp documentation for the specific backend rather than copying a generic command.

## 56.8 NVIDIA

Typical workflow:


In [ ]:
!cmake -B build-cuda -DGGML_CUDA=ON
!cmake --build build-cuda --config Release -j


Then:


In [ ]:
!build-cuda\bin\Release\llama-cli.exe -m "models\model.gguf" -ngl 99 -cnv


Verify CUDA is actually being used from the llama.cpp startup logs.

For NVIDIA systems, also inspect:


In [ ]:
!nvidia-smi


before troubleshooting llama.cpp.

Useful information:

```text
GPU name
VRAM usage
GPU utilization
temperature
driver version
```

## 56.9 AMD

Possible backends include:

```text
HIP / ROCm
Vulkan
```

depending on operating system and GPU.

Do not assume every AMD GPU supports every llama.cpp backend equally well.

Check:

[https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md)

before selecting the backend.

## 56.10 Intel

There are two very different situations.

### Intel integrated GPU

Examples:

```text
Intel UHD
Intel Iris Xe
```

These share system memory and usually have much less effective memory bandwidth than a dedicated GPU.

Start with CPU inference.

Then investigate:

```text
Vulkan
SYCL
```

if the hardware/software combination supports it.

### Intel Arc

Arc has dedicated VRAM and is much more interesting for local LLM inference.

Potential backends:

```text
Vulkan
SYCL
```

Use the current llama.cpp backend documentation to determine the best route for the particular Arc generation.

## 56.11 Apple Silicon

Apple Silicon uses unified memory:

```text
CPU
GPU
Neural/accelerator resources
        ↓
shared memory pool
```

This changes how memory should be evaluated.

For example, a Mac with:

```text
32 GB unified memory
```

cannot be compared directly with:

```text
8 GB dedicated GPU VRAM
```

because the architectures allocate memory differently.

llama.cpp's Metal backend is the normal route:


In [ ]:
!-DGGML_METAL=ON


Use the macOS build instructions rather than Windows commands.

## 56.12 CPU + GPU hybrid inference

A useful general model is:

```text
             Model
               |
        +------+------+
        |             |
       GPU           CPU
      VRAM          RAM
```

If the entire model fits comfortably on the GPU:

```text
GPU-heavy
→ usually best latency/throughput
```

If it does not:

```text
GPU + CPU
→ larger models become possible
→ performance may decrease
```

This is why a machine with:

```text
16 GB VRAM + 64 GB RAM
```

can sometimes run models that cannot fit entirely in 16 GB VRAM.

## 56.13 GPU selection for coding assistants

For a coding assistant, optimize for:

```text
1. Enough VRAM
2. Memory bandwidth
3. Backend support
4. Sustained performance
5. Model quality
6. Context capacity
7. Power/thermals
```

Do not optimize only for gaming FPS.

A GPU that is excellent for gaming but has insufficient VRAM can be frustrating for local LLM workloads.

## 56.14 Recommended hardware profiles

### Budget local AI

```text
CPU: modern 6-core+
RAM: 16–32 GB
GPU: integrated or 6–8 GB dedicated
Model: 3B–8B Q4
```

### Strong personal workstation

```text
CPU: modern 8-core+
RAM: 32–64 GB
GPU: 12–16 GB VRAM
Model: 7B–20B Q4/Q5
```

### High-end local AI workstation

```text
CPU: Ryzen 9 / Core i9 / equivalent
RAM: 64–128 GB
GPU: 24–32 GB VRAM
Model: 14B–32B Q4/Q5
```

### Large-model workstation

```text
RAM: 128–256+ GB
GPU: 48–80+ GB VRAM or multiple GPUs
Model: 32B–70B+
```

### Edge device

```text
ARM SBC
RAM: 8–16 GB
GPU: limited/integrated
Model: 1B–4B Q4
```

## 56.15 Final hardware rule

Before downloading a model, calculate:

```text
Does the model fit in VRAM?
        |
       YES
        ↓
Prefer high GPU offload

       NO
        |
        ↓
Does model + runtime fit in RAM?
        |
       YES
        ↓
Use CPU or CPU+GPU hybrid

       NO
        |
        ↓
Choose a smaller model or stronger machine
```

This should be the standard hardware-selection logic throughout the guide.
